In [ ]:
#new training code for final dataset(2020-2026)
#filtering features(only numerical) for training of the dataset
#Splitting train-test
#converted the model to time-series
#ARIMA work before SARIMAX
#Mainly model validation part

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("/Users/mangalyaphaye/Desktop/suvira_intelligence/data/final_2020_2026.csv")

df["date"] = pd.to_datetime(df["date"])

df.head()

,date,netWgt,fobvalue,price_usd_per_ton,chemical
0,2020-01-01,1.449699e+08,5.501491e+07,379.491888,Acetic Acid
1,2020-01-01,2.888552e+07,2.150078e+07,744.344399,Formates
2,2020-01-01,2.820466e+09,6.749782e+08,239.314451,KCL
3,2020-01-01,3.188438e+07,1.162156e+07,364.490829,PAC
4,2020-01-01,1.650663e+09,8.968455e+07,54.332438,Sulphur


In [2]:
df_ml = df[["date", "chemical", "price_usd_per_ton"]].copy()

df_ml = df_ml.sort_values(["chemical", "date"]).reset_index(drop=True)

df_ml.head()

,date,chemical,price_usd_per_ton
0,2020-01-01,Acetic Acid,379.491888
1,2020-02-01,Acetic Acid,408.298298
2,2020-03-01,Acetic Acid,383.695911
3,2020-04-01,Acetic Acid,383.981975
4,2020-05-01,Acetic Acid,400.561805


In [3]:
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

chemicals = df_ml["chemical"].unique()

results = []

for chem in chemicals:
    
    temp_df = df_ml[df_ml["chemical"] == chem].copy()
    temp_df = temp_df.sort_values("date")
    
    # TRAIN TEST SPLIT
    train = temp_df[temp_df["date"] <= "2025-09-01"].copy()
    test  = temp_df[(temp_df["date"] >= "2025-10-01") & 
                    (temp_df["date"] <= "2025-12-01")].copy()
    
    # Set index
    train.set_index("date", inplace=True)
    test.set_index("date", inplace=True)
    
    ts_train = train["price_usd_per_ton"].asfreq("MS")
    ts_test  = test["price_usd_per_ton"].asfreq("MS")
    
    ts_train = ts_train.dropna()
    ts_test  = ts_test.dropna()
    
    if len(ts_train) < 20 or len(ts_test) == 0:
        continue
    
    # Train ARIMA
    model = ARIMA(ts_train, order=(1,1,1))
    model_fit = model.fit()
    
    # Forecast test
    forecast = model_fit.forecast(steps=len(ts_test))
    
    # Evaluation
    mae = mean_absolute_error(ts_test, forecast)
    rmse = np.sqrt(mean_squared_error(ts_test, forecast))
    mape = np.mean(np.abs((ts_test - forecast) / ts_test)) * 100
    
    results.append({
        "Chemical": chem,
        "MAE": mae,
        "RMSE": rmse,
        "MAPE": mape
    })

/Users/mangalyaphaye/.pyenv/versions/suvira-env/lib/python3.11/site-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
/Users/mangalyaphaye/.pyenv/versions/suvira-env/lib/python3.11/site-packages/statsmodels/tsa/statespace/sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.'
/Users/mangalyaphaye/.pyenv/versions/suvira-env/lib/python3.11/site-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'


In [4]:
results_df = pd.DataFrame(results)

results_df = results_df.sort_values("MAPE")

results_df

,Chemical,MAE,RMSE,MAPE
1,Formates,92.282348,104.987712,6.684949
2,KCL,32.993765,48.866261,8.367556
6,Urea,50.125642,53.727800,12.145446
7,Water Treatment Chemicals,579.710296,657.050199,14.108433
0,Acetic Acid,68.843753,72.044456,17.338582
4,Sulphur,57.395320,57.819612,18.633826
5,Sulphuric Acid,30.818416,40.759106,20.174304
3,PAC,87.374456,112.615329,27.051244


In [5]:
results_df.to_csv("arima_results.csv", index=False)